# Phase 1: creating a db of article metadata

The JSON metadata file is updated weekly by ArXiv directly on Kaggle: https://www.kaggle.com/datasets/Cornell-University/arxiv

In [1]:
import duckdb
import json

In [7]:
# Read the first line of the JSON file we downloaded to get a better understanding of its structure
with open('data/raw/arxiv-metadata-oai-snapshot.json', 'r') as json_file:
    data = json.loads(json_file.readline())

print("Keys:", list(data.keys()))
print("\nSample values:")
data

Keys: ['id', 'submitter', 'authors', 'title', 'comments', 'journal-ref', 'doi', 'report-no', 'categories', 'license', 'abstract', 'versions', 'update_date', 'authors_parsed']

Sample values:


{'id': '0704.0001',
 'submitter': 'Pavel Nadolsky',
 'authors': "C. Bal\\'azs, E. L. Berger, P. M. Nadolsky, C.-P. Yuan",
 'title': 'Calculation of prompt diphoton production cross sections at Tevatron and\n  LHC energies',
 'comments': '37 pages, 15 figures; published version',
 'journal-ref': 'Phys.Rev.D76:013009,2007',
 'doi': '10.1103/PhysRevD.76.013009',
 'report-no': 'ANL-HEP-PR-07-12',
 'categories': 'hep-ph',
 'license': None,
 'abstract': '  A fully differential calculation in perturbative quantum chromodynamics is\npresented for the production of massive photon pairs at hadron colliders. All\nnext-to-leading order perturbative contributions from quark-antiquark,\ngluon-(anti)quark, and gluon-gluon subprocesses are included, as well as\nall-orders resummation of initial-state gluon radiation valid at\nnext-to-next-to-leading logarithmic accuracy. The region of phase space is\nspecified in which the calculation is most reliable. Good agreement is\ndemonstrated with data from th

## DuckDB creation

In [2]:
conn = duckdb.connect('data/arxiv_metadata.duckdb')

In [4]:
conn.execute("""
    CREATE TABLE staging AS
    SELECT * FROM read_json_auto('data/raw/arxiv-metadata-oai-snapshot.json');
""")

print(conn.execute("DESCRIBE staging").fetchdf())

print(conn.execute("SELECT * FROM staging LIMIT 1").fetchdf())

CatalogException: Catalog Error: Table with name "staging" already exists!

## TABLES

Let's created several tables that will allow us to naviguate through the metadata to get the papers (in PDF and LATEX for OCR benchmarking later) with the associated code.

**Architecture of the tables:**

papers — one row per arXiv paper, the central table:
- arxiv_id (PK, TEXT)
- title, abstract (TEXT)
- submitted_date, updated_date (DATE)
- primary_category (TEXT, e.g. cs.LG)
- doi (TEXT, nullable)
- comments (TEXT, nullable — author's free-text note, often where GitHub URLs live)
- license (TEXT)
- version_count (INT)
- github_url (TEXT, nullable — extracted by your regex)
- has_code (BOOLEAN — derived: TRUE if github_url is not null)
- pdf_path (TEXT, nullable — local filesystem path once downloaded)
- latex_source_path (TEXT, nullable — path to the .tar.gz)
- latex_extracted (BOOLEAN — did we successfully extract the source)

authors — one row per author, deduplicated by normalized name:
- author_id (INTEGER, auto)
- name_raw (TEXT)
- name_normalized (TEXT — e.g. lowercased, accents stripped — for joining)

paper_authors — the many-to-many link with order:
- arxiv_id (FK), 
- author_id (FK), 
- position (INT)
- PK is (arxiv_id, position)

paper_categories — papers can belong to multiple categories:
- arxiv_id (FK), 
- category (TEXT), 
- is_primary (BOOLEAN)

ingestion_runs — one row per pipeline execution, for auditability:
- run_id (auto), 
- source ('kaggle_dump' or 'arxiv_api'), 
- started_at, 
- completed_at, 
- n_papers, 
- n_files, 
- status, 
- notes

In [ ]:
# First table: papers 
conn.execute(r"""
CREATE OR REPLACE TABLE papers AS
SELECT
    id AS arxiv_id,
    title,
    abstract,

    -- versions is a LIST of STRUCTs like [{version: 'v1', created: 'Mon, 1 Jan 2023 ...'}, ...]
    -- DuckDB lists are 1-indexed (NOT 0-indexed like Python).
    -- The 'created' field is in RFC 2822 format, so we parse it with strptime.
    -- We treat 'GMT' as a literal because timezone parsing is finicky.
    strptime(versions[1].created, '%a, %d %b %Y %H:%M:%S GMT')::DATE
        AS submitted_date,

    -- update_date is already in YYYY-MM-DD format, just cast it
    update_date::DATE AS updated_date,

    -- categories is a space-separated string like 'cs.LG cs.AI stat.ML'
    -- split_part(text, ' ', 1) returns the FIRST piece
    split_part(categories, ' ', 1) AS primary_category,

    doi,
    comments,
    license,

    -- length() on a list returns its size
    length(versions) AS version_count,

    -- GitHub URL extraction:
    NULLIF(
        regexp_extract(
            COALESCE(abstract, '') || ' ' || COALESCE(comments, ''),
            'https?://github\.com/[\w\-\.]+/[\w\-\.]+'
        ),
        ''
    ) AS github_url,

    -- Derived boolean: TRUE if we found a GitHub link above.
    -- We compute it inline rather than as a generated column for portability.
    (NULLIF(
        regexp_extract(
            COALESCE(abstract, '') || ' ' || COALESCE(comments, ''),
            'https?://github\.com/[\w\-\.]+/[\w\-\.]+'
        ),
        ''
    ) IS NOT NULL) AS has_code,

    -- These three are placeholders. Phase 1 step 5 (the fetcher) populates them.
    NULL::VARCHAR AS pdf_path,
    NULL::VARCHAR AS latex_source_path,
    FALSE AS latex_extracted

FROM staging;
""")

# Quick sanity check
conn.execute("SELECT COUNT(*) AS n_papers, COUNT(github_url) AS n_with_code FROM papers").fetchdf()

,n_papers,n_with_code
0,3021763,71870


In [5]:
# ingestion_runs
conn.execute("""
CREATE OR REPLACE TABLE ingestion_runs (
    run_id INTEGER,
    source VARCHAR,
    started_at TIMESTAMP,
    completed_at TIMESTAMP,
    n_papers INTEGER,
    n_files INTEGER,
    status VARCHAR,
    notes VARCHAR
);
""")

In [8]:
# papers_categories
conn.execute("""
CREATE OR REPLACE TABLE paper_categories AS
SELECT 
    arxiv_id,
    category,
    (category = primary_category) AS is_primary
FROM (
    SELECT
        id AS arxiv_id,
        UNNEST (string_split(categories, ' ')) AS category,
        string_split(categories, ' ')[1] AS primary_category
    FROM staging           
);
""")

conn.execute("""
SELECT category, COUNT(*) AS n_papers
FROM paper_categories
GROUP BY category
ORDER BY n_papers DESC
LIMIT 50
""").fetchdf()

,category,n_papers
0,cs.LG,262731
1,hep-ph,195475
2,cs.CV,189805
3,hep-th,181703
4,quant-ph,178013
5,cs.AI,173990
6,gr-qc,121058
7,cond-mat.mtrl-sci,108299
8,cs.CL,107394
9,astro-ph,105380


In [6]:
# Step 1: explode authors_parsed with their position in the author list
conn.execute("""
CREATE OR REPLACE TABLE _exploded AS
SELECT
    arxiv_id,
    -- author_arr is a list like ['Smith', 'John P.', '']  (last, first, suffix)
    -- array_to_string joins it with spaces: 'Smith John P. '
    -- TRIM removes the trailing space
    TRIM(array_to_string(author_arr, ' ')) AS name_raw,
    -- Normalized form for joining: lowercase + accents stripped
    -- 'François' -> 'francois', so 'François Chollet' matches 'Francois Chollet'
    LOWER(strip_accents(TRIM(array_to_string(author_arr, ' ')))) AS name_normalized,
    position
FROM (
    SELECT
        id AS arxiv_id,
        author_arr,
        -- ROW_NUMBER assigns 1, 2, 3... within each paper, preserving UNNEST order
        ROW_NUMBER() OVER (PARTITION BY id) AS position
    FROM (
        SELECT id, UNNEST(authors_parsed) AS author_arr FROM staging
    )
);
""")

# Step 2: deduplicated authors table
conn.execute("""
CREATE OR REPLACE TABLE authors AS
SELECT
    ROW_NUMBER() OVER () AS author_id,
    name_raw,
    name_normalized
FROM (
    -- For each unique normalized name, pick any one of its raw spellings
    SELECT
        FIRST(name_raw) AS name_raw,
        name_normalized
    FROM _exploded
    GROUP BY name_normalized
);
""")

# Step 3: the link table
conn.execute("""
CREATE OR REPLACE TABLE paper_authors AS
SELECT
    e.arxiv_id,
    a.author_id,
    e.position
FROM _exploded e
JOIN authors a USING (name_normalized);
""")

# Drop the temp scratch table
conn.execute("DROP TABLE _exploded;")

# Sanity check: top 5 most prolific authors in the corpus
conn.execute("""
SELECT a.name_raw, COUNT(*) AS n_papers
FROM paper_authors pa
JOIN authors a USING (author_id)
GROUP BY a.name_raw
ORDER BY n_papers DESC
LIMIT 5
""").fetchdf()

,name_raw,n_papers
0,Zhang Y.,3331
1,Liu Yang,2792
2,Wang J.,2472
3,Wang Y.,2460
4,Wang Wei,2279


In [7]:
conn.execute("""
INSERT INTO ingestion_runs (
    run_id, source, started_at, completed_at, n_papers, n_files, status, notes
)
SELECT
    1,
    'kaggle_dump',
    CURRENT_TIMESTAMP,
    CURRENT_TIMESTAMP,
    (SELECT COUNT(*) FROM papers),
    0,
    'success',
    'Initial bulk load from Kaggle JSONL snapshot.'
""")

conn.execute("SELECT * FROM ingestion_runs").fetchdf()

,run_id,source,started_at,completed_at,n_papers,n_files,status,notes
0,1,kaggle_dump,2026-04-30 08:44:58.987391,2026-04-30 08:44:58.987391,3021763,0,success,Initial bulk load from Kaggle JSONL snapshot.


In [3]:
# Show all tables
conn.execute("SHOW TABLES").fetchdf()

# Check the size of staging vs the normalized tables
conn.execute("""
SELECT 'staging' AS tbl, COUNT(*) AS n FROM staging
UNION ALL SELECT 'papers', COUNT(*) FROM papers
UNION ALL SELECT 'paper_categories', COUNT(*) FROM paper_categories
UNION ALL SELECT 'authors', COUNT(*) FROM authors
UNION ALL SELECT 'paper_authors', COUNT(*) FROM paper_authors
""").fetchdf()

,tbl,n
0,staging,3021763
1,papers,3021763
2,paper_categories,5213855
3,authors,2337120
4,paper_authors,14292982


## Input list for fetching PDFs

The goal is to query our db to get 1000 recent papers on computer science for OCR and benchmarking the different open source algorithms to do so. 

In [ ]:
# Let's verify the distribution of papers across CS
conn.execute("""
            SELECT primary_category, COUNT(*) AS n_papers
            FROM papers
            WHERE primary_category LIKE 'cs.%' AND submitted_date >= '2024-01-01'
            GROUP BY primary_category
            ORDER BY n_papers DESC
""").fetchdf()

,primary_category,n_papers
0,cs.CV,80509
1,cs.LG,71441
2,cs.CL,48190
3,cs.RO,23191
4,cs.AI,21053
5,cs.CR,15522
6,cs.HC,11730
7,cs.SE,10627
8,cs.IT,8087
9,cs.IR,7526


First, let's acknolewdge the work of all these amazing scientists publishing so many papers in only 2 years, it is pretty impressive!

In [3]:
# Let's select the same number of papers across all 10 categories of CS:
conn.execute("""
             CREATE OR REPLACE TABLE benchmark_subset AS
             WITH ranked AS (
                 SELECT
                     arxiv_id,
                     title,
                     primary_category,
                     submitted_date,
                     has_code,
                     github_url,
                     ROW_NUMBER() OVER (
                         PARTITION BY primary_category
                         ORDER BY HASH(arxiv_id || '_seed42')
                     ) AS rn
                 FROM papers
                 WHERE primary_category IN (
                     'cs.CV', 'cs.LG', 'cs.CL', 'cs.RO', 'cs.AI',
                     'cs.CR', 'cs.HC', 'cs.SE', 'cs.IT', 'cs.IR'
                 )
                 AND submitted_date >= '2024-01-01'
                 AND submitted_date <= '2026-01-01'
             )
             SELECT
                 arxiv_id,
                 title,
                 primary_category,
                 submitted_date,
                 has_code,
                 github_url
             FROM ranked
             WHERE rn <= 100
             ORDER BY primary_category, rn;
             """)

conn.execute("""
SELECT primary_category, COUNT(*) AS n,
       SUM(CASE WHEN has_code THEN 1 ELSE 0 END) AS n_with_code
FROM benchmark_subset
GROUP BY primary_category
ORDER BY primary_category
""").fetchdf()

,primary_category,n,n_with_code
0,cs.AI,100,13.0
1,cs.CL,100,14.0
2,cs.CR,100,4.0
3,cs.CV,100,23.0
4,cs.HC,100,0.0
5,cs.IR,100,18.0
6,cs.IT,100,0.0
7,cs.LG,100,11.0
8,cs.RO,100,5.0
9,cs.SE,100,3.0


In [ ]:
conn.execute("""
    SELECT *
    FROM benchmark_subset
    LIMIT 30
""").fetchdf()

,arxiv_id,title,primary_category,submitted_date,has_code,github_url
0,2510.25612,Counterfactual-based Agent Influence Ranker fo...,cs.AI,2025-10-29,False,None
1,2508.01324,Towards Evaluation for Real-World LLM Unlearning,cs.AI,2025-08-02,False,None
2,2506.12483,MALM: A Multi-Information Adapter for Large La...,cs.AI,2025-06-14,False,None
3,2510.11143,Spec-Driven AI for Science: The ARIA Framework...,cs.AI,2025-10-13,False,None
4,2412.13422,Generating Diverse Hypotheses for Inductive Re...,cs.AI,2024-12-18,False,None
5,2410.01290,Towards a Law of Iterated Expectations for Heu...,cs.AI,2024-10-02,False,None
6,2501.08074,Artificial Liver Classifier: A New Alternative...,cs.AI,2025-01-14,False,None
7,2509.13347,OpenHA: A Series of Open-Source Hierarchical A...,cs.AI,2025-09-13,True,https://github.com/CraftJarvis/OpenHA
8,2512.16030,Do Large Language Models Know What They Don't ...,cs.AI,2025-12-17,False,None
9,2510.17052,ToolCritic: Detecting and Correcting Tool-Use ...,cs.AI,2025-10-19,False,None


In [4]:
conn.execute("""
    SELECT *
    FROM papers
    LIMIT 30
""").fetchdf()

,arxiv_id,title,abstract,submitted_date,updated_date,primary_category,doi,comments,license,version_count,github_url,has_code,pdf_path,latex_source_path,latex_extracted
0,0704.0001,Calculation of prompt diphoton production cros...,A fully differential calculation in perturba...,2007-04-02,2008-11-26,hep-ph,10.1103/PhysRevD.76.013009,"37 pages, 15 figures; published version",None,2,None,False,None,None,False
1,0704.0002,Sparsity-certifying Graph Decompositions,"We describe a new algorithm, the $(k,\ell)$-...",2007-03-31,2008-12-13,math.CO,None,To appear in Graphs and Combinatorics,http://arxiv.org/licenses/nonexclusive-distrib...,2,None,False,None,None,False
2,0704.0003,The evolution of the Earth-Moon system based o...,The evolution of Earth-Moon system is descri...,2007-04-01,2008-01-13,physics.gen-ph,None,"23 pages, 3 figures",None,3,None,False,None,None,False
3,0704.0004,A determinant of Stirling cycle numbers counts...,We show that a determinant of Stirling cycle...,2007-03-31,2007-05-23,math.CO,None,11 pages,None,1,None,False,None,None,False
4,0704.0005,From dyadic $\Lambda_{\alpha}$ to $\Lambda_{\a...,In this paper we show how to compute the $\L...,2007-04-02,2013-10-15,math.CA,None,None,None,1,None,False,None,None,False
5,0704.0006,Bosonic characters of atomic Cooper pairs acro...,We study the two-particle wave function of p...,2007-03-31,2015-05-13,cond-mat.mes-hall,10.1103/PhysRevA.75.043613,"6 pages, 4 figures, accepted by PRA",None,1,None,False,None,None,False
6,0704.0007,Polymer Quantum Mechanics and its Continuum Limit,A rather non-standard quantum representation...,2007-03-31,2008-11-26,gr-qc,10.1103/PhysRevD.76.044016,"16 pages, no figures. Typos corrected to match...",None,2,None,False,None,None,False
7,0704.0008,Numerical solution of shock and ramp compressi...,A general formulation was developed to repre...,2007-03-31,2009-02-05,cond-mat.mtrl-sci,10.1063/1.2975338,Minor corrections,http://arxiv.org/licenses/nonexclusive-distrib...,3,None,False,None,None,False
8,0704.0009,"The Spitzer c2d Survey of Large, Nearby, Inste...",We discuss the results from the combined IRA...,2007-04-02,2010-03-18,astro-ph,10.1086/518646,None,None,1,None,False,None,None,False
9,0704.0010,"Partial cubes: structures, characterizations, ...",Partial cubes are isometric subgraphs of hyp...,2007-03-31,2007-05-23,math.CO,None,"36 pages, 17 figures",None,1,None,False,None,None,False


In [5]:
# Final check of what has been downloaded after running fetch_files.py
conn.execute("""
    SELECT
        SUM(CASE WHEN pdf_path IS NOT NULL THEN 1 ELSE 0 END) AS done,
        SUM(CASE WHEN pdf_path IS NULL THEN 1 ELSE 0 END) AS remaining
    FROM benchmark_subset bs JOIN papers p USING (arxiv_id)
""").fetchdf()

,done,remaining
0,897.0,103.0


In [6]:
# After use, close the connection to our duckdb to avoid locking issues in future runs
conn.close()